<a href="https://colab.research.google.com/github/ThandiweM/msc-ai-thesis/blob/feature%2Fed5005-data-imbalance/ed5005_etivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Student name: Thandiwe Feziwe Mangana
#### Student ID: 24325104

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install polars
!pip install fg-data-profiling

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_recall_fscore_support
from sklearn import set_config
import pickle
import matplotlib.pyplot as plt
%matplotlib inline
from scipy.stats import loguniform, randint, uniform
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier
pd.set_option('display.max_columns', None)
from pathlib import Path
from data_profiling import ProfileReport
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import learning_curve
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import shap
from sklearn.model_selection import StratifiedGroupKFold

###Data loading and sampling

In [ ]:
# @title
class FeatureTaxanomy:
    """
    Contains features' schema

    """

    FEATURE_MAP = {
    'length_to':'length_to_address',
    'is_contract_creation':'coming_from_contract',
    'length_from':'length_from_address',
    'is_same_address':'from_is_same_as_to_address',
    'duration_seconds':'duration',
    'gas_used.gas_used_summation':'gas_used_summation',
    'gas_used.gas_used_average':'gas_used_average',
    'gas_used.gas_used_median':'gas_used_median',
    'gas_used.gas_used_standard_deviation':'gas_used_standard_deviation',
    'gas_used.gas_used_maximum_val':'gas_used_maximum_val',
    'gas_used.gas_used_minimum_val':'gas_used_minimum_val',
    'gas_used.gas_used_variance':'gas_used_variance',
    'gas_used.gas_used_range_value':'gas_used_range_value',
    'gas_used.gas_used_skewness':'gas_used_skewness',
    'gas_used.gas_used_mode':'gas_used_mode',
    'gas_used.gas_used_coefficient_of_variation':'gas_used_coefficient_of_variation',
    'gas_prices.gas_prices_summation':'gas_prices_summation',
    'gas_prices.gas_prices_average':'gas_prices_average',
    'gas_prices.gas_prices_median':'gas_prices_median',
    'gas_prices.gas_prices_standard_deviation':'gas_prices_standard_deviation',
    'gas_prices.gas_prices_maximum_val':'gas_prices_maximum_val',
    'gas_prices.gas_prices_minimum_val':'gas_prices_minimum_val',
    'gas_prices.gas_prices_variance':'gas_prices_variance',
    'gas_prices.gas_prices_range_value':'gas_prices_range_value',
    'gas_prices.gas_prices_skewness':'gas_prices_skewness',
    'gas_prices.gas_prices_mode':'gas_prices_mode',
    'gas_prices.gas_prices_coefficient_of_variation':'gas_prices_coefficient_of_variation',
    'cumulativeGasUsed.cumulativeGasUsed_summation':'cumulativeGasUsed_summation',
    'cumulativeGasUsed.cumulativeGasUsed_average':'cumulativeGasUsed_average',
    'cumulativeGasUsed.cumulativeGasUsed_median':'cumulativeGasUsed_median',
    'cumulativeGasUsed.cumulativeGasUsed_standard_deviation':'cumulativeGasUsed_standard_deviation',
    'cumulativeGasUsed.cumulativeGasUsed_maximum_val':'cumulativeGasUsed_maximum_val',
    'cumulativeGasUsed.cumulativeGasUsed_minimum_val':'cumulativeGasUsed_minimum_val',
    'cumulativeGasUsed.cumulativeGasUsed_variance':'cumulativeGasUsed_variance',
    'cumulativeGasUsed.cumulativeGasUsed_range_value':'cumulativeGasUsed_range_value',
    'cumulativeGasUsed.cumulativeGasUsed_skewness':'cumulativeGasUsed_skewness',
    'cumulativeGasUsed.cumulativeGasUsed_mode':'cumulativeGasUsed_mode',
    'cumulativeGasUsed.cumulativeGasUsed_coefficient_of_variation':'cumulativeGasUsed_coefficient_of_variation',
    'values.values_summation':'values_summation',
    'values.values_average':'values_average',
    'values.values_median':'values_median',
    'values.values_standard_deviation':'values_standard_deviation',
    'values.values_maximum_val':'values_maximum_val',
    'values.values_minimum_val':'values_minimum_val',
    'values.values_variance':'values_variance',
    'values.values_range_value':'values_range_value',
    'values.values_skewness':'values_skewness',
    'values.values_mode':'values_mode',
    'values.values_coefficient_of_variation':'values_coefficient_of_variation',
    'nonce.nonce_summation':'nonce_summation',
    'nonce.nonce_average':'nonce_average',
    'nonce.nonce_median':'nonce_median',
    'nonce.nonce_standard_deviation':'nonce_standard_deviation',
    'nonce.nonce_maximum_val':'nonce_maximum_val',
    'nonce.nonce_minimum_val':'nonce_minimum_val',
    'nonce.nonce_variance':'nonce_variance',
    'nonce.nonce_range_value':'nonce_range_value',
    'nonce.nonce_skewness':'nonce_skewness',
    'nonce.nonce_mode':'nonce_mode',
    'nonce.nonce_coefficient_of_variation':'nonce_coefficient_of_variation'}

    FEATURE_TO_KEEP = [
    'length_transaction_hash',
    'length_to_address',
    'coming_from_contract',
    'status',
    'log_removed',
    #'block_number',
    'gas_used',
    'length_from_address',
    'index',
    'gas_efficiency',
    'value',
    'chain_id',
    'message',
    'total_gas_cost',
    'gas_per_log_event',
    'log_index',
    'event_activity_flag',
    'normalized_token_transfer',
    'effective_gas_price',
    'cumulative_gas_used',
    'from_is_same_as_to_address',
    'gas_price_ratio',
    'token_transfer_amount',
    'length_log',
    'Error!',
    'log_count',
    'num_transaction',
    'duration',
    'number_of_errors',
    'error_rate',
    'gas_used_summation',
    'gas_used_average',
    'gas_used_median',
    'gas_used_standard_deviation',
    'gas_used_maximum_val',
    'gas_used_minimum_val',
    'gas_used_variance',
    'gas_used_range_value',
    'gas_used_skewness',
    'gas_used_mode',
    'gas_used_coefficient_of_variation',
    'gas_prices_summation',
    'gas_prices_average',
    'gas_prices_median',
    'gas_prices_standard_deviation',
    'gas_prices_maximum_val',
    'gas_prices_minimum_val',
    'gas_prices_variance',
    'gas_prices_range_value',
    'gas_prices_skewness',
    'gas_prices_mode',
    'gas_prices_coefficient_of_variation',
    'cumulativeGasUsed_summation',
    'cumulativeGasUsed_average',
    'cumulativeGasUsed_median',
    'cumulativeGasUsed_standard_deviation',
    'cumulativeGasUsed_maximum_val',
    'cumulativeGasUsed_minimum_val',
    'cumulativeGasUsed_variance',
    'cumulativeGasUsed_range_value',
    'cumulativeGasUsed_skewness',
    'cumulativeGasUsed_mode',
    'cumulativeGasUsed_coefficient_of_variation',
    'values_summation',
    'values_average',
    'values_median',
    'values_standard_deviation',
    'values_maximum_val',
    'values_minimum_val',
    'values_variance',
    'values_range_value',
    'values_skewness',
    'values_mode',
    'values_coefficient_of_variation',
    'nonce_summation',
    'nonce_average',
    'nonce_median',
    'nonce_standard_deviation',
    'nonce_maximum_val',
    'nonce_minimum_val',
    'nonce_variance',
    'nonce_range_value',
    'nonce_skewness',
    'nonce_mode',
    'nonce_coefficient_of_variation',
    'number_of_from_address',
    'number_of_unique_from_address',
    'number_of_to_address',
    'number_of_unique_to_address'
    ]

In [ ]:
# @title

class DataIngestion:
    """
    Contains functions for matrix manipulation namely:
    a : multiplication
    b : transpose
    c : inverse

    """

    base_path = ''
    fraud_path = ''
    legitimate_path = ''
    fraud_parquet_path = ''
    legitimate_parquet_path = ''

    def __init__(self, base_path, fraud_filename, legitimate_filename):

        """
        Constructor for the DataIngester class.
        It takes the parameters and generates file paths.

        Parameters
        ----------
        base_path : string
            The path to the folder in which all the data is stored.
        fraud_filename : string
            The name of the file containing the fraud data.
        legitimate_filename : string
            The name of the file containing the legitimate data.

        """
        self.base_path = base_path
        self.fraud_path = self.base_path + fraud_filename
        self.legitimate_path = self.base_path + legitimate_filename

        fraud_parquet_filename = 'fraud.parquet'
        self.fraud_parquet_path = self.base_path + fraud_parquet_filename
        legitimate_parquet_filename = 'legitimate.parquet'
        self.legitimate_parquet_path = self.base_path + legitimate_parquet_filename

    def ingest_data(self):

        """
        Ingests the raw data from the csv files and saves it as parquet files.

        This is to optimise for large csv files

        Parameters
        ----------
        None

        Returns
        -------
        tuple
            The paths to the generated and saved parquet files.

        """
        q_fraud = pl.scan_csv(self.fraud_path,
                       infer_schema_length=10000,
                       ignore_errors=True,
                       schema_overrides={'token_transfer_amount':pl.Float64})

        q_fraud.sink_parquet(self.fraud_parquet_path)

        q_legitimate = pl.scan_csv(self.legitimate_path,
                       infer_schema_length=10000,
                       ignore_errors=True,
                       schema_overrides={'token_transfer_amount':pl.Float64})

        q_legitimate.sink_parquet(self.legitimate_parquet_path)

        return self.fraud_parquet_path, self.legitimate_parquet_path

    def extract_subset(self, fraud_parquet_path, legitimate_parquet_path, n_samples=20000):

        """
        Extracts a subset of the data from the parquet files.
        It combines the fraud and legitimate data and saves it as a parquet file.

        Parameters
        ----------
        n_samples : int
            The number of samples to extract from each dataset.
            The default is 20000.
        fraud_parquet_path : string
            The path to the parquet file containing the fraud data.
        legitimate_parquet_path : string
            The path to the parquet file containing the legitimate data.

        Returns
        -------
        string
            The path to the generated and saved parquet file.

        """

        lazy_legitimate = pl.scan_parquet(legitimate_parquet_path)
        lazy_fraud = pl.scan_parquet(fraud_parquet_path)

        #Creating one dataset and extracting a sample of 20 000
        legitimate_count = lazy_legitimate.select(pl.len()).collect().item()
        fraud_count = lazy_fraud.select(pl.len()).collect().item()

        print(f"Normal transactions: {legitimate_count}")
        print(f"Fraud transactions: {fraud_count}")

        #Calculating the target for n_samples rows
        total_sample = n_samples
        fraud_ratio = fraud_count / (legitimate_count + fraud_count)
        target_fraud = int(total_sample * fraud_ratio)
        target_legitimate = total_sample - target_fraud

        #List of columns that have a mismatch
        mismatch_cols = ['erc_721_TokenAddress', 'erc_721_TokenName', 'erc_721_TokenSymbol']

        #Standardize both LazyFrames to String
        lazy_fraud = lazy_fraud.with_columns([
            pl.col(c).cast(pl.String) for c in mismatch_cols
        ])

        lazy_legitimate = lazy_legitimate.with_columns([
            pl.col(c).cast(pl.String) for c in mismatch_cols
        ])

        #Sampling from LazyFrames
        df_fraud = lazy_fraud.collect().sample(n=target_fraud, shuffle=True, seed=42)
        df_legitimate = lazy_legitimate.collect().sample(n=target_legitimate, shuffle=True, seed=42)

        #Concatenating and shuffling
        df_final = pl.concat([df_fraud, df_legitimate]).sample(fraction=1.0, shuffle=True, seed=42)

        #Save as parquet
        df_final.write_parquet(self.base_path + 'dataset.parquet')

        #Save as csv
        df_final.write_csv(self.base_path + 'dataset.csv')
        return self.base_path + 'dataset.parquet'

    from sklearn.model_selection import StratifiedGroupKFold

    def grouped_train_test_split(self, X, y, groups, n_splits=5, random_state=42):
      sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=random_state)
      train_idx, test_idx = next(sgkf.split(X, y, groups))

      return (X.iloc[train_idx], X.iloc[test_idx],
            y.iloc[train_idx], y.iloc[test_idx],
            groups.iloc[train_idx], groups.iloc[test_idx])


    def get_processed_data(self, dataset_parquet_path):

        """
        Loads the dataset from the parquet file and splits it into training and testing sets.
        It also renames the columns to more align with the feature taxonomy.

        Parameters
        ----------
        string: dataset_parquet_path
            The path to the parquet file containing the dataset.

        Returns
        -------
        tuple
            The X_train, X_test, y_train, y_test.

        """

        df = pd.read_parquet(dataset_parquet_path)
        y = df['flag']
        X = df.drop(columns=['flag'], axis=1)
        groups = df.loc[X.index, "address"]

        X_train, X_test, y_train, y_test, groups_train, groups_test = self.grouped_train_test_split(X, y, groups)
        #X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

        X_train = X_train.rename(columns=FeatureTaxanomy.FEATURE_MAP)
        X_test = X_test.rename(columns=FeatureTaxanomy.FEATURE_MAP)

      #   groups = df.loc[X.index, "address"]
      #   n_groups = groups.nunique()
      #   sizes = groups.value_counts()

      #   print(f"Rows: {len(groups)}")
      #   print(f"Unique wallets (groups): {n_groups}")
      #   print(f"Rows per wallet — mean: {sizes.mean():.1f}, "
      # f"median: {sizes.median():.0f}, max: {sizes.max()}")
      #   print(f"\nTop 10 biggest wallets:\n{sizes.head(10)}")
      #   print(f"\nShare of rows held by the top 1% of wallets: "

      # f"{sizes.head(max(1, n_groups // 100)).sum() / len(groups):.1%}")

        df = df.rename(columns=FeatureTaxanomy.FEATURE_MAP)
        wallet_cols = ['number_of_errors','error_rate','gas_used_summation','gas_used_average','gas_used_median','gas_used_standard_deviation','gas_used_maximum_val','gas_used_minimum_val','gas_used_variance',
                     'gas_used_range_value','gas_used_skewness','gas_used_mode','gas_used_coefficient_of_variation','gas_prices_summation','gas_prices_average','gas_prices_median','gas_prices_standard_deviation',
                     'gas_prices_maximum_val','gas_prices_minimum_val','gas_prices_variance','gas_prices_range_value','gas_prices_skewness','gas_prices_mode','gas_prices_coefficient_of_variation',
                     'cumulativeGasUsed_summation','cumulativeGasUsed_average','cumulativeGasUsed_median','cumulativeGasUsed_standard_deviation','cumulativeGasUsed_maximum_val','cumulativeGasUsed_minimum_val',
                     'cumulativeGasUsed_variance','cumulativeGasUsed_range_value','cumulativeGasUsed_skewness','cumulativeGasUsed_mode','cumulativeGasUsed_coefficient_of_variation','values_summation','values_average',
                     'values_median','values_standard_deviation','values_maximum_val','values_minimum_val','values_variance','values_range_value','values_skewness','values_mode','values_coefficient_of_variation','nonce_summation',
                     'nonce_average','nonce_median','nonce_standard_deviation','nonce_maximum_val','nonce_minimum_val','nonce_variance','nonce_range_value','nonce_skewness','nonce_mode','nonce_coefficient_of_variation','number_of_from_address',
                     'number_of_unique_from_address','number_of_to_address','number_of_unique_to_address', 'num_transaction','duration']

        n_fingerprints = df[wallet_cols].drop_duplicates().shape[0]
        n_addresses    = df['address'].nunique()

        print(f"Unique wallet-feature fingerprints: {n_fingerprints}")
        print(f"Unique addresses:                   {n_addresses}")   # expect 2115

        X_train = X_train[FeatureTaxanomy.FEATURE_TO_KEEP]
        X_test = X_test[FeatureTaxanomy.FEATURE_TO_KEEP]

        return X_train, X_test, y_train, y_test, groups_train, groups_test


In [ ]:
legitimate_parquet_path = "" #= '/content/drive/MyDrive/thesis_data/DeFiTransLyzer_legitimate.parquet'
fraud_parquet_path  = "" #= "/content/drive/MyDrive/thesis_data/DeFiTransLyzer_fraud.parquet"
dataset_path = '/content/drive/MyDrive/thesis_data/dataset.parquet'

data_ingestion = DataIngestion('/content/drive/MyDrive/thesis_data/', 'DeFiTransLyzer_fraud.csv', 'DeFiTransLyzer_legitimate.csv')
#fraud_parquet_path, legitimate_parquet_path = data_ingestion.ingest_data()
#dataset_path = data_ingestion.extract_subset(fraud_parquet_path, legitimate_parquet_path, n_samples = 50000)
X_train, X_test, y_train, y_test, groups_train, groups_test = data_ingestion.get_processed_data(dataset_path)

# print(f"X_train shape: {X_train.shape}")
# print(f"X_test shape: {X_test.shape}")
# print(f"y_test type: {type(y_test)}")
# print(f"y_train type: {type(y_train)}")

###Exploratory data analysis (EDA) and data preparation

In [ ]:

class DataPreparation:

    SKEWED_FEATURES = ['values_minimum_val','total_gas_cost','gas_used_minimum_val','value','nonce_minimum_val','values_variance']
    CATEGORICAL_FEATURES = ['event_activity_flag','chain_id','from_is_same_as_to_address']
    STANDARD_FEATURES =  ['length_transaction_hash','coming_from_contract','status','log_removed',#'block_number',
                          'gas_used', 'length_from_address','index','gas_efficiency', 'message','gas_per_log_event','log_index','normalized_token_transfer','effective_gas_price','cumulative_gas_used','gas_price_ratio',
        'token_transfer_amount','length_log','Error!','log_count','num_transaction','duration','number_of_errors','error_rate','gas_used_summation','gas_used_average','gas_used_median','gas_used_standard_deviation','gas_used_maximum_val','gas_used_variance','gas_used_range_value',
        'gas_used_skewness', 'gas_used_mode','gas_used_coefficient_of_variation','gas_prices_summation', 'gas_prices_average', 'gas_prices_median','gas_prices_standard_deviation','gas_prices_maximum_val','gas_prices_minimum_val','gas_prices_variance', 'gas_prices_range_value',
        'gas_prices_skewness','gas_prices_mode','gas_prices_coefficient_of_variation', 'cumulativeGasUsed_summation','cumulativeGasUsed_average','cumulativeGasUsed_median','cumulativeGasUsed_standard_deviation', 'cumulativeGasUsed_maximum_val', 'cumulativeGasUsed_minimum_val', 'cumulativeGasUsed_variance',
        'cumulativeGasUsed_range_value', 'cumulativeGasUsed_skewness','cumulativeGasUsed_mode','cumulativeGasUsed_coefficient_of_variation', 'values_summation', 'values_average','values_median', 'values_standard_deviation','values_maximum_val','values_range_value',
        'values_skewness','values_mode','values_coefficient_of_variation','nonce_summation','nonce_average','nonce_median', 'nonce_standard_deviation','nonce_maximum_val','nonce_variance','nonce_range_value', 'nonce_skewness','nonce_mode', 'nonce_coefficient_of_variation',
        'number_of_from_address','number_of_unique_from_address','number_of_to_address','number_of_unique_to_address' ]
    BINARIZE_FEATURES = ['length_to_address']


    @property
    def skewed_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.SKEWED_FEATURES))

    @property
    def standard_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.STANDARD_FEATURES))

    @property
    def categorical_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.CATEGORICAL_FEATURES))

    @property
    def binarize_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.BINARIZE_FEATURES))

    def update_features(self, features_to_drop):
      FeatureTaxanomy.FEATURE_TO_KEEP = list(set(FeatureTaxanomy.FEATURE_TO_KEEP) - set(features_to_drop))

    def plot_feature_by_fraud_flag(self, X_train, y_train, feature):
      #Check distribution of a feature by fraud status
      plt.figure(figsize=(10, 6))
      sns.boxplot(x=y_train, y=X_train[feature])
      plt.title(f'Distribution of {feature} by outcome')
      plt.show()

    def generate_profiling_report(self, X_train, report_path):
      profile_data = X_train.drop(columns=['df_index'], errors='ignore')
      profile = ProfileReport(profile_data, title="Thesis Dataset Profiling: Training Set",
                            correlations={
            "pearson": {"calculate": True},
            "spearman": {"calculate": False},
            "kendall": {"calculate": False},
            "phi_k": {"calculate": False},
        },
        interactions={"continuous": False})
      #Save the report as an HTML file
      profile.to_file(report_path)

    def generate_preprocessing_pipeline(self):

        clean_inf = FunctionTransformer(
            lambda X: np.where(np.isinf(X), np.nan, X),
            feature_names_out="one-to-one")

        enforce_finite = FunctionTransformer(
            lambda X: np.clip(np.nan_to_num(X, posinf=1e18, neginf=-1e18), -1e18, 1e18),
            feature_names_out="one-to-one")


        skew_pipeline = Pipeline(steps=[
        ('clean_inf', clean_inf),
        ('enforce_finite', enforce_finite),
        ('imputer', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ('scale', RobustScaler())])

        standard_pipeline = Pipeline(steps=[
        ('clean_inf', clean_inf),
        ('enforce_finite', enforce_finite),
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())])

        binarize_cat = FunctionTransformer(
            lambda X: (X == 0).astype(int),
            feature_names_out="one-to-one")

        binary_pipe = Pipeline(steps=[
            ('binarize', binarize_cat)])

        flag_pipeline = Pipeline(steps=[
        ('passthrough', FunctionTransformer(lambda X: X))])

        preprocess_pipeline = ColumnTransformer(
        transformers=[
            ('skew', skew_pipeline, self.skewed_features),
            ('standard', standard_pipeline, self.standard_features),
            ('binary', binary_pipe, self.binarize_features),
            ('flags', flag_pipeline, self.categorical_features)
        ],
        remainder="drop",
        verbose_feature_names_out=False)

        preprocess_pipeline.set_output(transform="pandas")
        return preprocess_pipeline


In [ ]:
X_train.head(5)

In [ ]:
data_preparation = DataPreparation()
#data_preparation.generate_profiling_report(X_train, '/content/drive/MyDrive/thesis_data/data_profiling_report.html')

In [ ]:
data_preparation.plot_feature_by_fraud_flag(X_train, y_train, 'total_gas_cost')

In [ ]:
data_preparation.plot_feature_by_fraud_flag(X_train, y_train, 'gas_used_minimum_val')

In [ ]:
data_preparation.plot_feature_by_fraud_flag(X_train, y_train,'value')

In [ ]:
data_preparation.plot_feature_by_fraud_flag(X_train, y_train,'nonce_minimum_val')

In [ ]:
data_preparation.plot_feature_by_fraud_flag(X_train, y_train,'values_variance')

In [ ]:
num_X = X_train.select_dtypes(include=[np.number])

inf_cols = num_X.columns[np.isinf(num_X).any()]
inf_cols.tolist()

In [ ]:
X_train['token_transfer_amount'].describe()

In [ ]:
X_train['length_to_address'].describe()

In [ ]:
features_to_drop = ['message', 'status', 'coming_from_contract', 'Error!', 'gas_price_ratio', 'gas_per_log_event','log_count', 'log_removed', 'log_index', 'length_log', 'normalized_token_transfer','length_from_address','length_transaction_hash']

data_preparation.update_features(features_to_drop)

X_train_u, X_test_u, y_train_u, y_test_u, groups_train_u, groups_test_u = data_ingestion.get_processed_data(dataset_path)

print(f"X_train shape: {X_train_u.shape}")
print(f"X_test shape: {X_test_u.shape}")
X_train_u.head(5)

In [ ]:
#data_preparation.generate_profiling_report(X_train, '/content/drive/MyDrive/thesis_data/data_profiling_report_processesed.html')

In [ ]:
pipe_test = data_preparation.generate_preprocessing_pipeline()
result = pipe_test.fit_transform(X_train_u)

In [ ]:
result['length_to_address'].describe()

In [ ]:
result['length_to_address'].value_counts()

In [ ]:
result

## Models comparison

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

def evaluate_model(X_test, y_test, model, model_name):
    #To ensure y_test is a flat 1D array to avoid dimension errors
    #.values handles the conversion, .ravel() ensures it is 1D
    y_true = y_test.values.ravel() if hasattr(y_test, 'values') else y_test

    #Get predictions
    y_probs = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    #Compute metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )

    #Compute PR-AUC and ROC-AUC
    pr_auc = average_precision_score(y_true, y_probs)
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    precision_array, recall_array, _ = precision_recall_curve(y_true, y_probs)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "fpr": fpr,
        "tpr": tpr,
        "Precision_array": precision_array,
        "Recall_array": recall_array
    }

    return results

def plot_pr_curves(results_list):
    plt.figure(figsize=(8, 6), dpi=300)

    for res in results_list:
        plt.plot(res['Recall_array'], res['Precision_array'],
                 label=f"{res['Model']} (AUC={res['PR-AUC']:.3f})")


    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curve comparison')

    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.7)
    #plt.savefig('pr_curve_comparison.pdf', bbox_inches='tight')
    pos_fraction = sum(y_test) / len(y_test)
    plt.axhline(y=pos_fraction, color='r', linestyle=':', label='No Skill')
    plt.show()

def plot_learning_curve_modifies(pipeline, X, y, groups, title, save_path=None):

    cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True,
                                       random_state=42)
    train_sizes, train_scores, val_scores = learning_curve(
        pipeline, X, y,
        groups=groups,
        cv=cv_strategy,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='average_precision',
        shuffle=True, random_state=42,
        error_score='raise',
    )

    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean,   val_std   = val_scores.mean(axis=1),   val_scores.std(axis=1)

    plt.figure(figsize=(10, 6), dpi=300)
    plt.title(f"Learning Curve: {title}")
    plt.xlabel("Training set size")
    plt.ylabel("PR-AUC")
    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.fill_between(train_sizes, train_mean - train_std,
                     train_mean + train_std, alpha=0.1, color="r")
    plt.plot(train_sizes, val_mean, 'o-', color="g", label="Cross-validation score")
    plt.fill_between(train_sizes, val_mean - val_std,
                     val_mean + val_std, alpha=0.1, color="g")
    plt.legend(loc="best")
    plt.grid(True)
    # if save_path:
    #     plt.savefig(save_path, bbox_inches="tight")   # .pdf for the thesis
    plt.show()


def plot_learning_curve(pipeline, X, y, title):

    plt.figure(figsize=(10, 6), dpi=300)
    train_sizes_ = np.linspace(0.2, 1.0, 5)

    #Calculate learning curve
    #Use the full pipeline so that preprocessing happens inside the CV loop
    train_sizes, train_scores, val_scores = learning_curve(
        pipeline, X, y, cv=5, n_jobs=-1,
        train_sizes=train_sizes_,
        scoring='f1',
        error_score='raise' # Helps identify if a split has 0 positive classes
    )

    #Calculate metrics
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    val_scores_mean = np.mean(val_scores, axis=1)
    val_scores_std = np.std(val_scores, axis=1)

    #Plot
    plt.figure(figsize=(10, 6))
    plt.title(f"Learning Curve: {title}")
    plt.xlabel("Training set size")
    plt.ylabel("F1 Score")

    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")

    plt.plot(train_sizes, val_scores_mean, 'o-', color="g", label="Cross validation score")
    plt.fill_between(train_sizes, val_scores_mean - val_scores_std,
                     val_scores_mean + val_scores_std, alpha=0.1, color="g")

    plt.legend(loc="best")
    plt.grid(True)
    plt.show()

In [ ]:
preprocess_pipeline = data_preparation.generate_preprocessing_pipeline()
# random forest pipeline
rf_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('rf', RandomForestClassifier(
         random_state=42,
         n_jobs=4
    ))
])

preprocess_pipeline.fit_transform(X_train_u)

In [ ]:
set_config(display="diagram")
rf_pipe

In [ ]:

param_distributions = {
    'rf__n_estimators':      randint(100, 500),        # more trees
    'rf__max_depth':         [None, 10, 20, 30, 50],   # Non
    'rf__min_samples_split': randint(2, 20),
    'rf__min_samples_leaf':  randint(1, 10),
    'rf__max_features':      ['sqrt', 0.3, 0.5],
}

#ensure balanced folds
#cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    rf_pipe,
    param_distributions,
    n_iter=20,
    scoring='average_precision', #Using PR-AUC instead of default accuracy
    n_jobs=-1,
    cv=cv_strategy,         #the stratified strategy
    error_score='raise',
    verbose=1               #for debuging
)

search.fit(X_train_u, y_train_u, groups = groups_train_u)
print("Best CV score = %0.3f:" % search.best_score_)
print("Best parameters: ", search.best_params_)

#store the best params and best model for later use
RF_best_params = search.best_params_
RF_best_model = search.best_estimator_

In [ ]:
std  = search.cv_results_['std_test_score'][search.best_index_]
print(f" CV PR-AUC = {search.best_score_:.4f} ± {std:.4f}")

In [ ]:
results = evaluate_model(X_test_u, y_test_u, RF_best_model, 'Random Forest')

print(f"f1 score = {results['F1-Score']}" )
print(f"PR Auc = {results['PR-AUC']}")
print(f"Roc Auc = {results['ROC-AUC']} ")
print(f"Precision = {results['Precision']}")
print(f"Recall = {results['Recall']}")

In [ ]:
plot_learning_curve_modifies(RF_best_model, X_train_u, y_train_u, groups_train_u, title= "Random Forest", save_path="figures/lc_rf.pdf")

In [ ]:
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

#XGBoost pipeline
xgb_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
     ('xgb', XGBClassifier(
         eval_metric='logloss',
         random_state=42,
         n_jobs=4,
         tree_method='hist'
    ))
])

#ensure balanced folds
#cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


#XGBoost grid
xgb_param_dist = {
    'xgb__n_estimators':      randint(100, 600),
    'xgb__max_depth':         randint(3, 11),
    'xgb__learning_rate':     loguniform(0.01, 0.3),
    'xgb__subsample':         uniform(0.6, 0.4),
    'xgb__colsample_bytree':  uniform(0.6, 0.4),
    'xgb__min_child_weight':  randint(1, 10)
}

search_xgb = RandomizedSearchCV(
    xgb_pipe,
    xgb_param_dist,
    n_iter=20,
    scoring='average_precision', #Using PR-AUC instead of default accuracy
    n_jobs=-1,
    cv=cv_strategy,         #the stratified strategy
    error_score='raise',
    verbose=1               #for debuging
)

search_xgb.fit(X_train_u, y_train_u, groups = groups_train_u)
print("Best CV score = %0.3f:" % search_xgb.best_score_)
print("Best parameters: ", search_xgb.best_params_)

#store the best params and best model for later use
XGB_best_params = search_xgb.best_params_
XGB_best_model = search_xgb.best_estimator_

In [ ]:
std  = search_xgb.cv_results_['std_test_score'][search_xgb.best_index_]
print(f" CV PR-AUC = {search_xgb.best_score_:.4f} ± {std:.4f}")

In [ ]:
results_xgb = evaluate_model(X_test_u, y_test_u, XGB_best_model, 'XGBoost')

print(f"f1 score = {results_xgb['F1-Score']}" )
print(f"PR Auc = {results_xgb['PR-AUC']}")
print(f"Roc Auc = {results_xgb['ROC-AUC']} ")
print(f"Precision = {results_xgb['Precision']}")
print(f"Recall = {results_xgb['Recall']}")

In [ ]:
plot_learning_curve_modifies(XGB_best_model, X_train_u, y_train_u, groups_train_u, "XGB Boost")

In [ ]:
from lightgbm import LGBMClassifier

#LightGBM pipeline
lgbm_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('lgbm', LGBMClassifier(
        verbose=-1,
        random_state=42,
        n_jobs=4,
    ))
])

lgbm_param_dist = {
    'lgbm__n_estimators':      randint(100, 600),
    'lgbm__num_leaves':        randint(20, 150),
    'lgbm__max_depth':         [-1, 6, 9, 12],
    'lgbm__learning_rate':     loguniform(0.01, 0.3),
    'lgbm__subsample':         uniform(0.6, 0.4),
    'lgbm__subsample_freq':    [1],
    'lgbm__colsample_bytree':  uniform(0.6, 0.4),
    'lgbm__min_child_samples': randint(5, 50),
}

cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

search_lgbm = RandomizedSearchCV(
    lgbm_pipe,
    lgbm_param_dist,
    n_iter=20,
    scoring='average_precision',
    n_jobs=-1,
    cv=cv_strategy,
    error_score='raise')

search_lgbm.fit(X_train_u, y_train_u, groups = groups_train_u)

print("Best CV score = %0.3f:" % search_lgbm.best_score_)
print("Best parameters: ", search_lgbm.best_params_)

#store the best params and best model for later use
LGBM_best_params = search_lgbm.best_params_
LGBM_best_model = search_lgbm.best_estimator_

In [ ]:
std  = search_lgbm.cv_results_['std_test_score'][search_lgbm.best_index_]
print(f" CV PR-AUC = {search_lgbm.best_score_:.4f} ± {std:.4f}")

In [ ]:
results_lgbm = evaluate_model(X_test_u, y_test_u, LGBM_best_model, 'LGBM Boost')

print(f"f1 score = {results_lgbm['F1-Score']}" )
print(f"PR Auc = {results_lgbm['PR-AUC']}")
print(f"Roc Auc = {results_lgbm['ROC-AUC']} ")
print(f"Precision = {results_lgbm['Precision']}")
print(f"Recall = {results_lgbm['Recall']}")

In [ ]:
plot_learning_curve_modifies(LGBM_best_model, X_train_u, y_train_u, groups_train_u, "LGBM Boost")

In [ ]:
#plot pr-auc
plot_pr_curves([results, results_xgb, results_lgbm])

## Imbalance handling comparison

In [ ]:

#LGBMClassifier SMOTE pipeline
smote_pipe = ImbPipeline([
    ('preprocessor', preprocess_pipeline),
    ('smote', SMOTE(random_state=42)),
    ('lgbm', LGBMClassifier(random_state=42))
])

search_smote_lgbm = RandomizedSearchCV(
    smote_pipe,
    lgbm_param_dist,
    n_iter=20,
    scoring='average_precision',
    n_jobs=1,
    cv=5,
    error_score='raise')
search_smote_lgbm.fit(X_train_u, y_train_u)

print("Best CV score = %0.3f:" % search_smote_lgbm.best_score_)
print("Best parameters: ",search_smote_lgbm.best_params_)

#store the best params and best model for later use
LGBM_smote_best_params = search_lgbm.best_params_
LGBM_smote_best_model = search_lgbm.best_estimator_

In [ ]:
results_lgbm_smote = evaluate_model(X_test_u, y_test_u, LGBM_smote_best_model, 'LGBM - SMOTE')
print(f"f1 score = {results_lgbm_smote['F1-Score']}" )
print(f"PR Auc = {results_lgbm_smote['PR-AUC']}")
print(f"Roc Auc = {results_lgbm_smote['ROC-AUC']} ")
print(f"Precision = {results_lgbm_smote['Precision']}")
print(f"Recall = {results_lgbm_smote['Recall']}")

In [ ]:
plot_learning_curve(smote_pipe, X_train_u, y_train_u, "LGBM Boost")

In [ ]:
legitimate_count = y_train.value_counts()[0]
fraud_count = y_train.value_counts()[1]
weight_ratio = legitimate_count / fraud_count

cost_sensitive_lgbm = LGBMClassifier(
    scale_pos_weight=weight_ratio,
    random_state=42
)

cost_sensitive_pipeline = Pipeline([
    ('preprocessor', preprocess_pipeline),
    ('lgbm', cost_sensitive_lgbm)
])

search_cost_sensitive_lgbm = RandomizedSearchCV(
    cost_sensitive_pipeline,
    lgbm_param_dist,
    n_iter=20,
    scoring='average_precision',
    n_jobs=1,
    cv=5,
    error_score='raise')

search_cost_sensitive_lgbm.fit(X_train_u, y_train_u)

print("Best CV score = %0.3f:" % search_cost_sensitive_lgbm.best_score_)
print("Best parameters: ",search_cost_sensitive_lgbm.best_params_)

#store the best params and best model for later use
LGBM_cost_sensitive_best_params = search_cost_sensitive_lgbm.best_params_
LGBM_cost_sensitive_best_model = search_cost_sensitive_lgbm.best_estimator_

In [ ]:
results_lgbm_cost_sensitive = evaluate_model(X_test_u, y_test_u, LGBM_cost_sensitive_best_model, 'LGBM - cost sensitive')
print(f"f1 score = {results_lgbm_cost_sensitive['F1-Score']}" )
print(f"PR Auc = {results_lgbm_cost_sensitive['PR-AUC']}")
print(f"Roc Auc = {results_lgbm_cost_sensitive['ROC-AUC']} ")
print(f"Precision = {results_lgbm_cost_sensitive['Precision']}")
print(f"Recall = {results_lgbm_cost_sensitive['Recall']}")

In [ ]:
#Transform training data so SHAP sees what the model sees
X_train_transformed = lgbm_pipe.named_steps['preprocess'].transform(X_train)

best_pipeline = search_lgbm.best_estimator_

best_booster = best_pipeline.named_steps['lgbm'].booster_
#fit the explainer on the transformed data
explainer = shap.TreeExplainer(best_booster)
shap_values = explainer.shap_values(X_train_transformed)

shap.summary_plot(shap_values, X_train_transformed)

In [ ]:
#Calculate mean absolute SHAP value for every column
importance_df = pd.DataFrame({
    'feature': X_train_transformed.columns,
    'importance': np.abs(shap_values).mean(0)
}).sort_values(by='importance', ascending=False)

print(importance_df)

print("Least important features:")
print(importance_df.tail(10))

In [ ]:

#Sort values for the plot
df_sorted = importance_df.sort_values(by='importance', ascending=False).reset_index(drop=True)

#Plot the importance values
plt.figure(figsize=(10, 6))
plt.plot(df_sorted.index, df_sorted['importance'], marker='o', linestyle='-', markersize=4)

#Add labels and title
plt.title('Feature Importance Elbow Plot')
plt.xlabel('Number of Features Included')
plt.ylabel('Mean Absolute SHAP Value')
plt.grid(True)

plt.axvline(x=20, color='r', linestyle='--', label='Pruning Threshold (20)')
plt.legend()

plt.show()

In [ ]:

importance_summary = importance_df.copy()

threshold = 0.015
importance_summary['is_near_zero'] = importance_summary['importance'] < threshold

total_features = len(importance_summary)
zero_count = importance_summary['is_near_zero'].sum()

print(f"Total Features: {total_features}")
print(f"Features with near-zero importance (< {threshold}): {zero_count}")
print("\nBottom 20 features (importance values):")
print(importance_summary.tail(20))


In [ ]:
features_to_drop = importance_summary[importance_summary['is_near_zero']]['feature'].tolist()

print(f"Number of features to drop: {len(features_to_drop)}")
print("Features to be dropped:", features_to_drop)

In [ ]:
data_preparation.update_features(features_to_drop)

X_train_s, X_test_s, y_train_s, y_test_s = data_ingestion.get_processed_data(dataset_path)

X_train_s.shape

In [ ]:
preprocess_pipeline = data_preparation.generate_preprocessing_pipeline()

#LightGBM pipeline
lgbm_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('lgbm', LGBMClassifier(verbose=-1))
])


preprocess_pipeline.fit_transform(X_train_s)

#LightGBM grid
lgbm_param_dist = {
    'lgbm__n_estimators': randint(50, 200),
    'lgbm__max_depth': randint(3, 10),
    'lgbm__learning_rate': uniform(0.01, 0.3),
    'lgbm__num_leaves': randint(20, 50)
}

search_lgbm = RandomizedSearchCV(
    lgbm_pipe,
    lgbm_param_dist,
    n_iter=20,
    scoring='average_precision',
    n_jobs=1,
    cv=5,
    error_score='raise')

search_lgbm.fit(X_train_s, y_train_s)

print("Best CV score = %0.3f:" % search_lgbm.best_score_)
print("Best parameters: ", search_lgbm.best_params_)

#store the best params and best model for later use
LGBM_best_params = search_lgbm.best_params_
LGBM_best_model = search_lgbm.best_estimator_

In [ ]:
results_lgbm_smote = evaluate_model(X_test_s, y_test_s, LGBM_best_model, 'LGBM - SHAP')
print(f"f1 score = {results_lgbm_smote['F1-Score']}")
print(f"PR Auc = {results_lgbm_smote['PR-AUC']}")
print(f"Roc Auc = {results_lgbm_smote['ROC-AUC']} ")
print(f"Precision = {results_lgbm_smote['Precision']}")
print(f"Recall = {results_lgbm_smote['Recall']}")

In [ ]:
plot_learning_curve( LGBM_best_model, X_train_s, y_train_s, "LGBM -SHAP")